
# ⚡ Energy Forecasting with XGBoost: Line-by-Line Explanation

This notebook provides a **comprehensive explanation** of how to build, evaluate, and visualize an XGBoost regression model using household power consumption features.

---

### 📦 Installing and Importing Required Libraries

```python
!pip install xgboost --quiet
```
* Installs the `xgboost` library if it is not already available in the environment.

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
```
* Imports libraries for:
  - **`pandas`**: loading and handling tabular data
  - **`numpy`**: numerical computations
  - **`matplotlib.pyplot`**: plotting and visualization
  - **`xgboost`**: gradient boosting algorithm for supervised learning

---

### 📥 Load Dataset

```python
df = pd.read_csv('data/features.csv')
```
* Reads the dataset containing features and target values for energy forecasting.

```python
X = df.drop(columns=['Global_active_power', 'datetime'])
y = df['Global_active_power']
```
* `X`: contains input features (excluding the target and timestamp).
* `y`: the target variable, `Global_active_power`, which we want to predict.

---
### 🔀 Train-Test Split

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
```
* Splits data into 80% training and 20% testing sets.
* `shuffle=False` preserves the temporal order for time-series consistency.

---

### 🧱 Create DMatrix for XGBoost

```python
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)
```
* Converts the dataset into XGBoost’s optimized internal data structure `DMatrix`, which is faster and more efficient for training.

---

### ⚙️ Set XGBoost Parameters

```python
params = {
    'objective': 'reg:squarederror',
    'max_depth': 6,
    'eta': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}
```

| Parameter | Description |
|-----------|-------------|
| `objective` | Regression task using squared error loss |
| `max_depth` | Maximum tree depth (controls model complexity) |
| `eta` | Learning rate (step size shrinkage) |
| `subsample` | Proportion of training samples used per boosting round |
| `colsample_bytree` | Fraction of features used per tree |
| `seed` | Random seed for reproducibility |

---

* Uses **RMSE** as the evaluation metric.
---

### 🏋️ Train Final Model

```python
# Train model with evaluation history
#num_rounds = 5  # Number of boosting rounds (training time/epoch)
#model = xgb.train(params, dtrain, num_rounds)

evals_result = {}  # Dictionary to store evaluation results
num_rounds = 100   #Number of boosting rounds (training time/epoch)

model = xgb.train(params,dtrain,num_boost_round=num_rounds,
    evals=[(dtrain, 'train'), (dtest, 'test')],evals_result=evals_result,verbose_eval=10)  # verbose_eval=10 means Print evaluation every 10 rounds
```

---

### 💾 Save the Model

```python
model.save_model('xgb_energy_model.json')
```
* Saves the trained XGBoost model in JSON format for future reuse or deployment.

---
### 4. Predict on test data
y_pred = model.predict(dtest)

### 5. Evaluate performance (e.g., RMSE)
```python
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test RMSE: {rmse:.4f}")
```
### 📈 Visualize Learning Curve

```python
# 5. Plot learning curve
plt.figure(figsize=(10, 5))
plt.plot(evals_result['train']['rmse'], label='Train RMSE')
plt.plot(evals_result['test']['rmse'], label='Test RMSE')
plt.title('XGBoost Learning Curve')
plt.xlabel('Boosting Rounds')
plt.ylabel('RMSE')
plt.legend()
plt.grid(True)
plt.show()
```
* Plots RMSE over boosting rounds for both training and validation sets.
* Helps identify underfitting or overfitting based on the divergence of curves.

---

### 🧠 Feature Importance Plot

```python
xgb.plot_importance(model, max_num_features=10)
plt.title('Top 10 Feature Importances')
plt.show()
```
* Displays the top 10 most important features based on their contribution to prediction.
* Helps interpret model decisions and prioritize influential variables.

---

### ✅ Summary

This notebook demonstrates:
1. Loading and preparing data.
2. Initializing and tuning an XGBoost regressor.
3. Evaluating via train test split.
4. Visualizing model performance and feature importance.
